In [1]:
%%capture 
%run ./minimal_blockchain.ipynb

# CRYSTALS-Dilithium

The official submission for CRYSTALS-Dilithium to NIST: https://pq-crystals.org/dilithium/data/dilithium-specification-round3-20210208.pdf

A python package is available: https://pypi.org/project/dilithium/#description. dilithium provides capabilites for generating public-private key pairs, signing and verifying messages.

The official design defines three versions of Dilithium corresponding to the NIST Security Level: 2, 3, and 5. Each version has different parameter values, as well as output size (public key size and signature size) and performance (Gen, Sign, and Verify). 

In [2]:
pip install dilithium

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
# from dilithium.dilithium import Dilithium2 as Dilithium # Dilithium 2
# from dilithium.dilithium import Dilithium3 as Dilithium # Dilithium 3
from dilithium.dilithium import Dilithium5 as Dilithium # Dilithium 5

In [14]:
class DilithiumProvider(CryptoProvider):
    """
    DilithiumProvider implements the CRYSTALS-Dilithium cryptographic operations for key generation,
    signing, signature verification, and hashing using the post-quantum secure Dilithium method.
    """

    def __repr__(self) -> str:
        return "DilithiumProvider"

    def generate_key_pair(self) -> Tuple[bytes, bytes]:
        """
        Generate a Dilithium key pair.

        Returns:
            Tuple[bytes, bytes]: A tuple containing the private and public keys as bytes.
        """
        key_seed = os.urandom(32)
        public_key, private_key  = Dilithium.keygen(key_seed)
        return private_key, public_key

    def sign(self, private_key: bytes, message: str) -> bytes:
        """
        Sign a message using the provided Dilithium private key.

        Args:
            private_key (bytes): The private key used to sign the message.
            message (str): The message to be signed.

        Returns:
            bytes: The generated signature as bytes.
        """
        message_bytes = message.encode('utf-8')
        signature = Dilithium.sign_with_input(private_key, message_bytes)
        return signature

    def verify(self, public_key: bytes, message: str, signature: bytes) -> bool:
        """
        Verify a signature for a given message and Dilithium public key.

        Args:
            public_key (bytes): The public key used to verify the signature.
            message (str): The message whose signature is to be verified.
            signature (bytes): The signature to verify.

        Returns:
            bool: True if the signature is valid; False otherwise.
        """
        message_bytes = message.encode('utf-8')
        try:
            Dilithium.verify(public_key, message_bytes, signature)
            return True
        except Exception:
            return False

    def hash(self, data: str) -> str:
        """
        Compute a cryptographic hash of the provided data using SHA-256.

        Args:
            data (str): The data to be hashed.

        Returns:
            str: The resulting hash as a hexadecimal string.
        """
        hasher = hashlib.sha256()
        hasher.update(data.encode('utf-8'))
        return hasher.hexdigest()

In [15]:
crypto_provider = DilithiumProvider()
private_key1, public_key1 = crypto_provider.generate_key_pair()
address1 = base64.b64encode(public_key1).decode("utf-8")
private_key2, public_key2 = crypto_provider.generate_key_pair()
address2 = base64.b64encode(public_key2).decode("utf-8")

transaction1 = Transaction(address1, address2, 30, crypto_provider)
transaction1.sign_transaction(private_key1)

transaction2 = Transaction(address2, address1, 10, crypto_provider)
transaction2.sign_transaction(private_key2)

transaction3 = Transaction(address2, address1, 15, crypto_provider)
transaction3.sign_transaction(private_key2)

blockchain = Blockchain(2, 3, crypto_provider)
blockchain.add_transaction(transaction1)
blockchain.add_transaction(transaction2)
blockchain.add_transaction(transaction3)
blockchain.mine_pending_transactions()

print(blockchain)

print("\nChecking the balances of the addresses...")
print(f"Balance of address1: {blockchain.get_balance(address1)}")
print(f"Balance of address2: {blockchain.get_balance(address2)}")

blockchain.mine_pending_transactions()
print("\nMining the last pending transaction...")
print("Checking the balances of the addresses again...")
print(f"Balance of address1: {blockchain.get_balance(address1)}")
print(f"Balance of address2: {blockchain.get_balance(address2)}")

print("\nChecking the validity of the blockchain...")
print(f"Blockchain validity: {blockchain.is_valid()}")

blockchain.chain[1].transactions[1].amount = 20
print("Modifying blockchain...")
print(f"Blockchain validity: {blockchain.is_valid()}")

tr =  b'\xee\x9c7,\x89\xcc\xd2\xc3\xe6Cf\x0b\xc1\xd5i$?6\xectQ\xfe\xd7\x8eX\x194\xb21\x05\xcdC'
m =  b'u1bvaZlynHbEH1DlXXByjZ6U2582HNvoY8+qkTqgrF6cL3LsrVjqGd1wxJVEzE1gucKuwutS3cwscWS4448dwA77Y/HwT/3z3FDg+ZPGpKms//+K2u/gI45+ba0quPtsuHnaGbTMliQMh7mob429OM9YWolzkTa89at9QLK4B4A6Izsh1jBiJ4fg/Iy0VEOTWu0/eo70sGNYeOQCGbyZDzQSFz7a6tUXVNVS1ylTjbNCmhzTTyjieM+xMoQutrE0u0kbqe1MxZn9ew4vG6P7GXx8RUSSmS8QQ4oPssseE0xD0qYgHSXiuPbvGQn7Y5pI1wh4gLwndZCIVgEQYx+enpoiTMK8C0aQtJgNs/YF3oR+vkLwKIq6LYMUYJuzqz7zWQHT6ywcVpo5kWWK9ctdXzD2x3HoPRKY59w4K9DSyj35SUTLifJv4LIb1LDmNws48aruN8TdF+o3jHW+7HG5Fam8nYPK/VGH1pTxRGKgy/eAhrTGRvlw9NoS4zLb6uegikHw5cYfw1kcWAMQj29Sumu1667q907F5hq9E5WTWhq3CACK9g8oTwNzt20IHvb47/nCB+I1QM7Mub0AWPMxsy0PUMNDe3AqRfxjToqjMJKjR3nls6/k+zSNGIUPM4B7nbcHuBor+Lxat2PGDNEfSC9GFgiPjujNLIj30BxWhfINdaeq2i17+UGtklxzvaFCyemeat7M2v8+ojyPI2+jpVegq/XvtWvo95dTO6rEVIGhTFZgka0cV1hYfS+BKA6iAlLk9Ht+CL3P5DWtAA3eWF3pqBtPz2IxQUzLybDMkA7jJ6+287Z2/oeHbXoGI8M8tPh49bNP5DhI5XAeP4Z8Hqvu8jb1I5ocUae0ke6i6QEsvo63MPJ3Rn3Dw0NwP2eGV

tr =  b'vv\x03\xa6\xe0=x\xb5T\xe5\xa2n7\xfb\x86\xf2(\x1e\xa9\r\xdb\x8c\x8a\x02_\x8d\x96o\xa8\xe48\x07'
m =  b'Pfe6E30WTpDg56j13nIYLEHwvUnDZHi79DwthV7vChaVzXJ3C0L+YGHd8M94y29r2tj2X9ljvz9TvqwdlYaGqeAX7XA1fxkUS9mMhzgfanK+LOFiup/kkby2oiclJT/gD7E1tHAbeEeWwyW7d7xfbwLaCTXTHPEnKcoogVrzUN8j/IG6leU1Tk/fEEDns8UOJXQv3289RsTQdWImxEfT/nTYfqtwohzn4rL8VSoVFYvLd3mZuWEBTDQ/G1wsaDl5UAgtGiM7oRWMDU25siRiN6iPvwT0FN82SESVFCEakkzKUziKgaGKiGFYjrJokjSWfBg37UeQpyWwzVNty8b+f79A+O34MKCP8u36oVRXLVhfRddf5j+NPRnAKGQ+HTM4DTxcaR/Wpf4zqtqhyd8nKTanD48OKnvQN80jQvp6mp2gX1TqxNO0B0rNqYEqYVDsqFIQxCuLRZh6IVyv0j4VPybcoNYimyHBfdmdvanCs3X4sb57Cj0hpwxDFLSNwdkLjz2rCtYVDcqsxQqA//aR0fTm95EYHpWVp66pX1klYie+RY/Uxp2M1aZRd+XtJcSQwRJfg+G3dQHauoxQcd8rgB99E8PmDRk+una+muI4WSsorqoz7QmJyCl3sOYYagXY3UM+1L6TSYsS+ZFuRAj5FHUlSyuNtS1Y9h0NNIoCjBtp6+EEQ9+IEJiywGcbImdZ+0R2qROX35hBTOKpECZC4mYtcr+URmSQ92ICiSZFvKcKUkgAtSZsOYkv1ehEiN2xcMHJybjT24BbC72+Yj/Zm0DJIZfefb/+3xjW0Co7+cl4e2rSUrSmFSOuyM9k2uOt3XQvSsMSGGxsImIBUwrbBwUf102eaeloNGxRt2DRqV7KcYDAPa2kQPC0lo


Mining the last pending transaction...
Checking the balances of the addresses again...
Balance of address1: -5
Balance of address2: 5

Checking the validity of the blockchain...
Blockchain validity: True
Modifying blockchain...
Blockchain validity: True
